# Setup & Initialization

In [1]:
import random
import pyspark
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
from datetime import datetime
import ConnectionConfig as cc
from importlib import reload

reload(cc)
from pyspark.sql.functions import *

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

/home/jovyan/work/02-treasure-hunt-dwh/ConnectionConfig.py
Running in Docker environment.
Setting JAVA_HOME to /usr/lib/jvm/java-17-openjdk-arm64
SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter-data4
LANGUAGE: C.UTF-8
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-arm64
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
PYTHONPATH: /usr/local/spark/python/lib/py4j-0.10.9.9-src.zip:/usr/local/spark/python:
_CE_CONDA: 
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/s

In [2]:
spark = cc.startLocalCluster("dimUserIncremental")
spark.getActiveSession()

Running in Docker: using pre-installed Spark + JARs


## Reading the source table

In [3]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

jdbc:postgresql://postgres-data4:5432/catchem?user=postgres&password=data&ssl=false


In [4]:
# Load DataFrames
df_operational_user = cc.read_postgres_without_partition(spark, "user_table")
df_operational_treasure = cc.read_postgres_without_partition(spark, "treasure")
df_operational_treasure_log = cc.read_postgres_by_date(spark, "treasure_log", "log_time")
df_operational_city = cc.read_postgres_without_partition(spark, "city")
df_operational_country = cc.read_postgres_without_partition(spark, "country")

# Transforming the source to the dimension format

### Selecting and renaming columns for easier joins and transformations; also, we can filter or aggregate here if needed to optimize later steps

In [5]:
df_operational_city = df_operational_city.select(
    "city_id",
    "city_name",
    F.col("country_code").alias("country_id")
)

df_operational_country = df_operational_country.select(
    F.col("code").alias("country_id"),
    F.col("name").alias("country_name")
)

df_operational_user = df_operational_user.select(
    F.col("id").alias("user_id"),
    F.col("city_city_id").alias("city_id")
)

df_operational_treasure_log = df_operational_treasure_log \
    .groupBy(F.col("hunter_id").alias("user_id")) \
    .agg(F.count("*").alias("treasure_found"))

df_operational_treasure = df_operational_treasure \
    .select(F.col("owner_id").alias("user_id")) \
    .distinct()

print('user size: ', df_operational_user.count())
print('treasure size: ', df_operational_treasure.count())
print('treasure_log size: ', df_operational_treasure_log.count())
print('city size: ', df_operational_city.count())
print('country size: ', df_operational_country.count())

user size:  486523
treasure size:  33830
treasure_log size:  486340
city size:  1264588
country size:  249


In [6]:
# Join location data (Inner join usually fine for City -> Country)
df_location = df_operational_city.join(df_operational_country, "country_id")

# Prepare the "Dedicator" flag (Avoid distinct here to save performance; join handles it)
df_is_dedicator = df_operational_treasure.withColumn("dedicator", F.lit(True))

# Main Transformation
df_transformed = df_operational_user \
    .join(df_location, "city_id", "left") \
    .join(df_operational_treasure_log, "user_id", "left") \
    .join(df_is_dedicator, "user_id", "left") \
    .select(
    "user_id",
    "city_name",
    "country_name",
    # Use coalesce to handle nulls for users with no logs/flags
    F.coalesce(F.col("treasure_found"), F.lit(0)).cast("integer").alias("treasure_found"),
    F.coalesce(F.col("dedicator"), F.lit(False)).alias("dedicator")
) \
    .withColumn("Experience_level",
                F.when(F.col("treasure_found") == 0, "Starter")
                .when(F.col("treasure_found") < 4, "Amateur")
                .when(F.col("treasure_found") <= 10, "Professional")
                .otherwise("Pirate")
                ) \
    .withColumn("scd_start", F.current_timestamp()) \
    .withColumn("scd_end", F.lit(None).cast("timestamp")) \
    .withColumn("current", F.lit(True))

df_transformed = df_transformed.drop("treasure_found")

# Incremental Update (SCD Type 2)

### Detect Changes

In [7]:
# Setup
run_timestamp = F.current_timestamp()
dt_dim_user = DeltaTable.forPath(spark, "spark-warehouse/dim_user")
df_current = dt_dim_user.toDF().filter(F.col("current") == True)

# Join to find changes
# We join the new transformed data against the current active records in DWH
detected_changes = df_transformed.alias("source").join(
    df_current.alias("dwh"),
    on="user_id",
    how="left"
).filter(
    (F.col("dwh.user_id").isNull()) |  # New Record
    (F.col("dwh.city_name") != F.col("source.city_name")) |
    (F.col("dwh.country_name") != F.col("source.country_name")) |
    (F.col("dwh.experience_level") != F.col("source.experience_level")) |
    (F.col("dwh.dedicator") != F.col("source.dedicator"))
).select(
    "source.*",
    F.col("dwh.user_id").alias("dwh_user_id"),
    F.col("dwh.city_name").alias("dwh_city_name"),
    F.col("dwh.country_name").alias("dwh_country_name"),
    F.col("dwh.dedicator").alias("dwh_dedicator"),
    F.col("dwh.experience_level").alias("dwh_experience_level"),
    F.col("dwh.scd_start").alias("dwh_scd_start")
)

In [8]:
detected_changes.show(truncate=False)

+-------------------------------------------------+---------+------------+---------+----------------+--------------------------+-------+-------+-------------------------------------------------+-------------+----------------+-------------+--------------------+-------------------------+
|user_id                                          |city_name|country_name|dedicator|Experience_level|scd_start                 |scd_end|current|dwh_user_id                                      |dwh_city_name|dwh_country_name|dwh_dedicator|dwh_experience_level|dwh_scd_start            |
+-------------------------------------------------+---------+------------+---------+----------------+--------------------------+-------+-------+-------------------------------------------------+-------------+----------------+-------------+--------------------+-------------------------+
|[00 00 1C BF A8 E5 46 C1 BA A6 21 DE 39 F4 E2 D4]|Palanga  |Lithuania   |false    |Professional    |2026-06-10 15:14:05.384561|NULL   |tru

### Prepare Upserts (The Union)

In [9]:
# Create the Insert records (New version of data)
df_inserts = detected_changes.select(
    "user_id", "city_name", "country_name",
    "dedicator", "experience_level"
).withColumn("scd_start", run_timestamp) \
    .withColumn("scd_end", F.lit(None).cast("timestamp")) \
    .withColumn("current", F.lit(True))

# Create the Update records (To close existing active records)
df_updates = detected_changes.filter(F.col("dwh_user_id").isNotNull()).select(
    F.col("user_id"),
    F.col("dwh_city_name").alias("city_name"),
    F.col("dwh_country_name").alias("country_name"),
    F.col("dwh_dedicator").alias("dedicator"),
    F.col("dwh_experience_level").alias("experience_level"),
    F.col("dwh_scd_start").alias("scd_start")
).withColumn("scd_end", run_timestamp) \
    .withColumn("current", F.lit(False))

# Combine them
df_upserts = df_inserts.unionByName(df_updates)
# .unionByName() solves this by acting like a smart schema-matcher. It looks at the actual column names (e.g., user_id, scd_end, treasure_found) and aligns them perfectly, regardless of what order they appear in either DataFrame

df_upserts.show(truncate=False)

+-------------------------------------------------+---------+------------+---------+----------------+--------------------------+--------------------------+-------+
|user_id                                          |city_name|country_name|dedicator|experience_level|scd_start                 |scd_end                   |current|
+-------------------------------------------------+---------+------------+---------+----------------+--------------------------+--------------------------+-------+
|[00 00 1C BF A8 E5 46 C1 BA A6 21 DE 39 F4 E2 D4]|Palanga  |Lithuania   |false    |Professional    |2026-06-10 15:14:15.647203|NULL                      |true   |
|[00 00 1C BF A8 E5 46 C1 BA A6 21 DE 39 F4 E2 D4]|Leyssard |France      |false    |Amateur         |2026-05-01 08:36:02.17951 |2026-06-10 15:14:15.647203|false  |
+-------------------------------------------------+---------+------------+---------+----------------+--------------------------+--------------------------+-------+



### The Delta Merge

In [10]:
dt_dim_user.alias("target").merge(
    source=df_upserts.alias("updates"),
    condition="target.user_id = updates.user_id AND target.current = true AND updates.current = false"
).whenMatchedUpdate(set={
    "scd_end": "updates.scd_end",
    "current": "updates.current"
}).whenNotMatchedInsert(values={
    "user_id": "updates.user_id",
    "city_name": "updates.city_name",
    "country_name": "updates.country_name",
    "dedicator": "updates.dedicator",
    "experience_level": "updates.experience_level",
    "scd_start": "updates.scd_start",
    "scd_end": "updates.scd_end",
    "current": "updates.current"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### History

In [11]:
# To see the file-level history of changes
dt_dim_user.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-----------------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation   |operationParameters                                                                                                                                                                                                       |
+-------+-----------------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2      |2026-06-10 15:14:37.78 |MERGE       |{predicate -> ["(((user_id#85 = user_id#34) AND current#93) AND NOT current#417)"], matchedPredicates -> [{"actionType":"update"}], notMatchedPredicates -> 

## Cleaning up the environment (stop Spark and reset variables)

In [12]:
spark.stop()

In [13]:
%reset